# imports

In [1]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# connection and load

In [2]:
db_path = '../../database/financial_data.duckdb'
conn = duckdb.connect(db_path)

In [3]:
ASSET = 'BTC'
INTERVAL = '1h'

In [6]:
query = f"""
    SELECT * 
    FROM gold_crypto_features 
    WHERE asset_symbol = '{ASSET}' 
    AND interval = '{INTERVAL}' 
    ORDER BY date ASC
"""

# dataframe

In [7]:
df = conn.execute(query).df()

# datetime index 

In [8]:
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

In [9]:
df.head(3)

,asset_symbol,asset_class,exchange,interval,open,high,low,close,volume,daily_volatility,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-02 17:00:00,BTC,Crypto,Bybit,1h,6908.5,7230.5,6908.5,7081.5,9009.751,322.0,...,0.024733,0.045471,0.537267,6908.5,1552.213,6911.0,6782.5,6.380255e+07,NaN,NaN
2020-04-02 18:00:00,BTC,Crypto,Bybit,1h,7081.5,7085.0,6729.5,6787.0,4520.872,355.5,...,-0.042477,0.052380,0.161744,7081.5,9009.751,7230.5,6908.5,3.068316e+07,NaN,NaN
2020-04-02 19:00:00,BTC,Crypto,Bybit,1h,6787.0,6863.5,6723.0,6796.5,1274.324,140.5,...,0.001399,0.020672,0.523132,6787.0,4520.872,7085.0,6729.5,8.660943e+06,NaN,NaN


# dropping cols and cleaning

In [10]:
metadata_cols = ['asset_symbol', 'asset_class', 'exchange', 'interval']

In [11]:
df = df.drop(columns=metadata_cols, errors='ignore')
print(f"Columns remaining: {len(df.columns)}")

Columns remaining: 47


In [12]:
df.head(5)

,open,high,low,close,volume,daily_volatility,sma_7,sma_30,rsi_14,macd,...,log_returns,hl_ratio,close_position,prev_close,prev_volume,prev_high,prev_low,turnover,open_interest,funding_rate
date,,,,,,,,,,,,,,,,,,,,,
2020-04-02 17:00:00,6908.5,7230.5,6908.5,7081.5,9009.751,322.0,6790.000000,6524.483333,85.656499,130.772825,...,0.024733,0.045471,0.537267,6908.5,1552.213,6911.0,6782.5,6.380255e+07,NaN,NaN
2020-04-02 18:00:00,7081.5,7085.0,6729.5,6787.0,4520.872,355.5,6805.857143,6543.566667,58.856604,123.436911,...,-0.042477,0.052380,0.161744,7081.5,9009.751,7230.5,6908.5,3.068316e+07,NaN,NaN
2020-04-02 19:00:00,6787.0,6863.5,6723.0,6796.5,1274.324,140.5,6826.571429,6563.550000,59.298990,117.040547,...,0.001399,0.020672,0.523132,6787.0,4520.872,7085.0,6729.5,8.660943e+06,NaN,NaN
2020-04-02 20:00:00,6796.5,6799.0,6644.0,6738.0,1411.383,155.0,6840.500000,6580.166667,55.352126,106.028688,...,-0.008645,0.023004,0.606452,6796.5,1274.324,6863.5,6723.0,9.509899e+06,NaN,NaN
2020-04-02 21:00:00,6738.0,6843.0,6711.5,6825.5,1056.962,131.5,6847.642857,6600.233333,59.675391,103.172917,...,0.012902,0.019266,0.866920,6738.0,1411.383,6799.0,6644.0,7.214294e+06,NaN,NaN


# check with new crypto-cols

In [15]:
new_cols = ['turnover', 'open_interest', 'funding_rate']
available_new = [c for c in new_cols if c in df.columns]
missing_new = [c for c in new_cols if c not in df.columns]

print(f"Columns available: {available_new}")
print(f"Columns MISSING:    {missing_new}")

Columns available: ['turnover', 'open_interest', 'funding_rate']
Columns MISSING:    []


In [16]:
for c in available_new:
    coverage = (1 - df[c].isna().mean()) * 100
    print(f"  {c}: {coverage:.1f}% non-null ({df[c].notna().sum():,} / {len(df):,} rows)")

  turnover: 100.0% non-null (53,638 / 53,638 rows)
  open_interest: 95.1% non-null (51,031 / 53,638 rows)
  funding_rate: 3.0% non-null (1,599 / 53,638 rows)


# Drop funding_rate — only ~3% coverage, not usable for ML

In [17]:
if 'funding_rate' in df.columns:
    df = df.drop(columns=['funding_rate'])
    print("Dropped funding_rate — not suitable for model training")
else:
    print("funding_rate not in columns, nothing to drop")

Dropped funding_rate — not suitable for model training


# new features, eda found oi pct_change spikes predict 24× baseline next-hour returns

In [ ]:
if 'open_interest' in df.columns:
    df['oi_change_1p']  = df['open_interest'].pct_change(1)
    df['oi_change_5p']  = df['open_interest'].pct_change(5)
    df['oi_change_20p'] = df['open_interest'].pct_change(20)
        
    oi_roll_mean = df['open_interest'].rolling(50).mean()
    oi_roll_std  = df['open_interest'].rolling(50).std()
    df['oi_zscore_50'] = (df['open_interest'] - oi_roll_mean) / oi_roll_std
    
    df = df.drop(columns=['open_interest'])
